In [1]:
import pandas as pd

In [4]:
# helper functions
def count_table(series, order):
    out = series.value_counts().reindex(order, fill_value=0).rename("count").to_frame()
    n = out["count"].sum()
    out["percent"] = (out["count"] / n * 100).round(1) if n else 0.0
    return out
    
def factor_table(subset):
    n = len(subset)
    rows = []
    for col, label in factor_cols.items():
        count = int(subset[col].sum())
        rows.append({
            "factor": label,
            "count": count,
            "percent": round(100 * count / n, 1) if n else 0.0,
        })
    return pd.DataFrame(rows)

In [5]:
# load dataset
df = pd.read_csv("../data/feedback_quality.csv")
df.head()

,evaluator,pair,preference,preference_strength,preferred_condition,preference_factor_correctness,preference_factor_specificity,preference_factor_actionability,preference_factor_appropriateness
0,1,1,A,Mild,Treatment,0,1,0,0
1,1,2,Same,Same,Same,0,0,0,0
2,1,3,B,Mild,Treatment,0,1,0,0
3,1,4,A,Mild,Treatment,0,1,0,0
4,1,5,A,Mild,Treatment,0,1,0,0


In [6]:
# field mapping 
condition_order = ["Treatment", "Control", "Same"]
strength_order = ["Strong", "Mild"]
factor_cols = {
    "preference_factor_correctness": "Correctness",
    "preference_factor_specificity": "Specificity",
    "preference_factor_actionability": "Actionability",
    "preference_factor_appropriateness": "Pedagogical appropriateness",
}
treatment = df[df["preferred_condition"] == "Treatment"]
control = df[df["preferred_condition"] == "Control"]

In [7]:
print("Overall preference counts")
print(count_table(df["preferred_condition"], condition_order))

Overall preference counts
                     count  percent
preferred_condition                
Treatment               65     61.9
Control                  4      3.8
Same                    36     34.3


In [8]:
print("Preference counts by evaluator")
print(
    pd.crosstab(df["evaluator"], df["preferred_condition"])
    .reindex(columns=condition_order, fill_value=0)
)

Preference counts by evaluator
preferred_condition  Treatment  Control  Same
evaluator                                    
1                           17        4    14
2                           24        0    11
3                           24        0    11


In [9]:
# preference strength
print(f"Preference strength when treatment was preferred (n={len(treatment)})")
print(count_table(treatment["preference_strength"], strength_order))

print(f"Preference strength when control was preferred (n={len(control)})")
print(count_table(control["preference_strength"], strength_order))

Preference strength when treatment was preferred (n=65)
                     count  percent
preference_strength                
Strong                  10     15.4
Mild                    55     84.6
Preference strength when control was preferred (n=4)
                     count  percent
preference_strength                
Strong                   0      0.0
Mild                     4    100.0


In [10]:
# # preference factors
print(f"Factors among treatment preferences (n={len(treatment)})")
print(factor_table(treatment).to_string(index=False))

print(f"Factors among control preferences (n={len(control)})")
print(factor_table(control).to_string(index=False))

Factors among treatment preferences (n=65)
                     factor  count  percent
                Correctness      8     12.3
                Specificity     59     90.8
              Actionability      7     10.8
Pedagogical appropriateness     10     15.4
Factors among control preferences (n=4)
                     factor  count  percent
                Correctness      0      0.0
                Specificity      4    100.0
              Actionability      0      0.0
Pedagogical appropriateness      0      0.0
